# 6th attempt - RCNN

In [37]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [38]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [39]:
SIZE = 10
AMOUNT_BOARDS = 1000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2

In [40]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 41366
len train:  33506
len val:  3723
len test:  4137


In [41]:
import tensorflow as tf
import numpy as np

# --- פרמטרים ---
SIZE = 10             # גודל הלוח
gen_data = gen - 1    # מספר הלוחות הרציפים בקלט
BATCH_SIZE = 32
EPOCHS = 3

# --- PREPROCESSING ---
# X_train_array.shape = (num_samples + gen_data, SIZE, SIZE, 1)
# y_train_array.shape = (num_samples, 1)  ← תא אחד בלבד (0 או 1)

num_samples = X_train_array.shape[0] - gen_data

X_train = np.zeros((num_samples, gen_data, SIZE, SIZE, 1), dtype='float32')
y_train = np.zeros((num_samples, 1), dtype='float32')  # רק תא אחד

for i in range(num_samples):
    X_train[i] = X_train_array[i:i+gen_data]   # רצף של gen_data לוחות
    y_train[i] = y_train_array[i]              # הפלט: תא אחד (0/1)

print("X_train shape:", X_train.shape)  # (num_samples, gen_data, SIZE, SIZE, 1)
print("y_train shape:", y_train.shape)  # (num_samples, 1)

# --- MODEL ---
model = tf.keras.Sequential([
    tf.keras.layers.ConvLSTM2D(
        filters=32,
        kernel_size=(3,3),
        activation='relu',
        padding='same',
        return_sequences=True,
        input_shape=(gen_data, SIZE, SIZE, 1)
    ),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.ConvLSTM2D(
        filters=64,
        kernel_size=(3,3),
        activation='relu',
        padding='same',
        return_sequences=False
    ),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # ← פלט יחיד בינארי
])

X_train shape: (33505, 1, 10, 10, 1)
y_train shape: (33505, 1)


c:\Users\דרור\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [42]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d_6 (ConvLSTM2D)      │ (None, 1, 10, 10, 32)  │        38,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 1, 10, 10, 32)  │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_7 (ConvLSTM2D)      │ (None, 10, 10, 64)     │       221,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 10, 10, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       409,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 669,697 (2.55 MB)

 Trainable params: 669,505 (2.55 MB)

 Non-trainable params: 192 (768.00 B)

In [43]:
# אימון
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 106s 92ms/step - accuracy: 0.8173 - loss: 0.4007 - val_accuracy: 0.8348 - val_loss: 0.3624
Epoch 2/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 137s 164ms/step - accuracy: 0.8550 - loss: 0.3255 - val_accuracy: 0.8367 - val_loss: 0.3581
Epoch 3/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 78s 93ms/step - accuracy: 0.8838 - loss: 0.2714 - val_accuracy: 0.8302 - val_loss: 0.3882
Epoch 4/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 77s 86ms/step - accuracy: 0.9152 - loss: 0.2029 - val_accuracy: 0.8248 - val_loss: 0.4307
Epoch 5/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 70s 83ms/step - accuracy: 0.9571 - loss: 0.1118 - val_accuracy: 0.8232 - val_loss: 0.6146
Epoch 6/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 89s 106ms/step - accuracy: 0.9780 - loss: 0.0612 - val_accuracy: 0.8239 - val_loss: 0.7251
Epoch 7/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 163s 127ms/step - accuracy: 0.9850 - loss: 0.0413 - val_accuracy: 0.8245 - val_loss: 0.8577
Epoch 8/10
838/838 ━━━━━━━━━━━━━━━━━━━━ 111s 132ms/step - accuracy: 0.9879 - loss: 0

In [44]:
X_test = X_test_array.reshape((-1, gen_data, SIZE, SIZE, 1)).astype('float32')
y_test = y_test_array.astype('float32')

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

  1/130 ━━━━━━━━━━━━━━━━━━━━ 50s 389ms/step - accuracy: 0.8125 - loss: 1.9810

130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.8365 - loss: 1.1765
Test accuracy: 0.836


In [45]:
import numpy as np
from sklearn.metrics import confusion_matrix



In [46]:
# יצירת סט בדיקה
num_samples_test = X_test_array.shape[0] - gen_data
X_test = np.zeros((num_samples_test, gen_data, SIZE, SIZE, 1), dtype='float32')
y_test = np.zeros((num_samples_test, 1), dtype='float32')

for i in range(num_samples_test):
    X_test[i] = X_test_array[i:i+gen_data]
    y_test[i] = y_test_array[i]

# הפעלת פונקציית ההערכה
results = evaluate_model(model, X_test, y_test)


130/130 ━━━━━━━━━━━━━━━━━━━━ 20s 128ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        437 │        431 │
│ True=Dead    │        246 │       3022 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.836
Precision   : 0.640
Recall      : 0.503
F1-score    : 0.564
